# nanoTriton — GPU Validation

**Run on Google Colab with a T4 GPU: Runtime → Change runtime type → T4 GPU**

Pipeline: Python `@kernel` → typed IR → PTX → runs on GPU, validated against NumPy.

In [ ]:
# Step 1: Clone/pull repo and install CuPy
# After this cell finishes, go to Runtime → Restart session, then run all remaining cells
import os
if not os.path.exists('nanoTriton'):
    !git clone https://github.com/MaruthiV/nanoTriton.git
else:
    !cd nanoTriton && git pull
!pip install cupy-cuda12x -q
print('Done. Now go to Runtime → Restart session, then run the cells below.')

In [ ]:
# Step 2: Run this after restarting the session
import sys
sys.path.insert(0, 'nanoTriton')

import cupy as cp
import numpy as np
import gpucc

cc = cp.cuda.Device(0).compute_capability
sm = f'sm_{cc}'
print('gpucc loaded successfully')
print(f'GPU: compute capability {cc}  → compiling for {sm}')

## Part 1: matmul

Naive matrix multiplication — each thread computes one element of C = A @ B.

In [ ]:
from gpucc import kernel, float32, int32, N, M, K
from gpucc.runtime.cupy_runner import GPURunner

@kernel
def matmul(A: float32[M, K], B: float32[K, N], C: float32[M, N],
           m: int32, k: int32, n: int32):
    row = block_id(0) * block_size(0) + thread_id(0)
    col = block_id(1) * block_size(1) + thread_id(1)
    acc = 0.0
    for ki in range(k):
        acc = acc + A[row, ki] * B[ki, col]
    C[row, col] = acc

print('── IR ──────────────────────────────────────────────────')
print(matmul.ir_text)

ptx_matmul = matmul.compile(opt_level=2, sm_version=sm)
runner_matmul = GPURunner(ptx_matmul, 'matmul')

MM, KK, NN = 64, 64, 64
np_A = np.random.rand(MM, KK).astype(np.float32)
np_B = np.random.rand(KK, NN).astype(np.float32)

A_gpu = cp.array(np_A)
B_gpu = cp.array(np_B)
C_gpu = cp.zeros((MM, NN), dtype=np.float32)

BLOCK2 = (16, 16)
GRID2  = ((MM + BLOCK2[0] - 1) // BLOCK2[0],
          (NN + BLOCK2[1] - 1) // BLOCK2[1])

runner_matmul(A_gpu, B_gpu, C_gpu,
              np.int32(MM), np.int32(KK), np.int32(NN),
              grid=GRID2, block=BLOCK2)

np.testing.assert_allclose(C_gpu.get(), np_A @ np_B, rtol=1e-4, atol=1e-4)
print('✓ matmul output matches NumPy @ reference')

## Part 2: vector_add

Compute `c[i] = a[i] + b[i]` for 1M elements.

In [ ]:
from gpucc import kernel, float32, int32, N
from gpucc.runtime.cupy_runner import GPURunner

@kernel
def vector_add(a: float32[N], b: float32[N], c: float32[N], n: int32):
    tid = block_id(0) * block_size(0) + thread_id(0)
    if tid < n:
        c[tid] = a[tid] + b[tid]

print('── IR ──────────────────────────────────────────────────')
print(vector_add.ir_text)

ptx_va = vector_add.compile(opt_level=2, sm_version=sm)
runner_va = GPURunner(ptx_va, 'vector_add')

N_ELEMS = 1 << 20
np_a = np.random.rand(N_ELEMS).astype(np.float32)
np_b = np.random.rand(N_ELEMS).astype(np.float32)

a_gpu = cp.array(np_a)
b_gpu = cp.array(np_b)
c_gpu = cp.zeros(N_ELEMS, dtype=np.float32)

BLOCK = 256
GRID  = (N_ELEMS + BLOCK - 1) // BLOCK

runner_va(a_gpu, b_gpu, c_gpu, np.int32(N_ELEMS), grid=(GRID,), block=(BLOCK,))

np.testing.assert_allclose(c_gpu.get(), np_a + np_b, rtol=1e-5)
print('✓ vector_add output matches NumPy reference')

In [ ]:
import time

N_RUNS = 100
cp.cuda.Stream.null.synchronize()
t0 = time.perf_counter()
for _ in range(N_RUNS):
    runner_va(a_gpu, b_gpu, c_gpu, np.int32(N_ELEMS), grid=(GRID,), block=(BLOCK,))
cp.cuda.Stream.null.synchronize()
t1 = time.perf_counter()

elapsed_ms = (t1 - t0) / N_RUNS * 1000
bw_gb_s = (N_ELEMS * 3 * 4) / (elapsed_ms / 1000) / 1e9
print(f'Average time per kernel: {elapsed_ms:.3f} ms')
print(f'Effective memory bandwidth: {bw_gb_s:.1f} GB/s  (T4 peak: ~300 GB/s)')

## Part 3: Optimization passes

Constant folding + dead code elimination — before vs after.

In [ ]:
from gpucc import kernel, float32, int32, N

@kernel
def scaled_add(a: float32[N], b: float32[N], c: float32[N], n: int32):
    tid = block_id(0) * block_size(0) + thread_id(0)
    scale = 2.0 * 3.0   # constant expression — folded to 6.0 at compile time
    if tid < n:
        c[tid] = (a[tid] + b[tid]) * scale

ptx_raw = scaled_add.compile(opt_level=0, sm_version=sm)
ptx_opt = scaled_add.compile(opt_level=2, sm_version=sm)

print('── PTX without optimization ──────────────────────────')
print(ptx_raw[:800], '...')
print('── PTX with opt_level=2 ──────────────────────────────')
print(ptx_opt[:800], '...')

if 'mul.f32' in ptx_raw:
    print('✓ Unoptimized: 2.0 * 3.0 emits a mul.f32 instruction')
if 'mul.f32' not in ptx_opt:
    print('✓ Optimized: constant folded — no multiply in PTX')